In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')
from src import functions as fc

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

In [3]:
df_modeling = fc.load_data_clean("bank_final.csv")

Buscando archivo en: /Users/jbp/Desktop/IRONHACK/SEMANA7/ML_project/data/cleaned/bank_final.csv


In [49]:
columnas_features = ["previous", "poutcome_success", "euribor3m", "cons.price.idx", "pdays", "marital_single"]

features = df_modeling[columnas_features]
target = df_modeling["target"]

In [50]:
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=0)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)

x_test_scaled = scaler.transform(x_test)

Decision Tree:

In [62]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=6)

In [63]:
tree.fit(x_train_scaled, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,6
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [64]:
pred = tree.predict(x_test_scaled)

In [ ]:
accuracy_tree = tree.score(x_test_scaled, y_test) 
print(f"La precisión del modelo es: {accuracy_tree:.2f}")

La precisión del modelo es: 0.65


Chequeo:

In [66]:
tree_importance = {feature : importance for feature, importance in zip(x_train.columns, tree.feature_importances_)}
tree_importance 

{'previous': np.float64(0.020714024035735506),
 'poutcome_success': np.float64(0.035676427726904644),
 'euribor3m': np.float64(0.6676322352906034),
 'cons.price.idx': np.float64(0.21686272245815094),
 'pdays': np.float64(0.021389843886053413),
 'marital_single': np.float64(0.037724746602552005)}

Randomize Search:

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist_tree = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None] + list(range(3, 30)),
    'min_samples_split': randint(2, 50),
    'min_samples_leaf': randint(1, 20),
    'max_features': [None, 'sqrt', 'log2'],
    'class_weight': [None, 'balanced']}

random_tree = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(),
    param_distributions=param_dist_tree,
    n_iter=60,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=2,
    random_state=42)

random_tree.fit(x_train_scaled, y_train)

Fitting 5 folds for each of 60 candidates, totalling 300 fits
[CV] END class_weight=None, criterion=gini, max_depth=16, max_features=log2, min_samples_leaf=8, min_samples_split=22; total time=   0.0s
[CV] END class_weight=None, criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=11, min_samples_split=12; total time=   0.0s
[CV] END class_weight=None, criterion=gini, max_depth=16, max_features=log2, min_samples_leaf=8, min_samples_split=22; total time=   0.0s
[CV] END class_weight=None, criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=11, min_samples_split=12; total time=   0.0s
[CV] END class_weight=None, criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=11, min_samples_split=12; total time=   0.0s
[CV] END class_weight=None, criterion=gini, max_depth=16, max_features=log2, min_samples_leaf=8, min_samples_split=22; total time=   0.0s
[CV] END class_weight=None, criterion=gini, max_depth=16, max_features=log2, min_samples_leaf=8, m

,estimator,DecisionTreeClassifier()
,param_distributions,"{'class_weight': [None, 'balanced'], 'criterion': ['gini', 'entropy', ...], 'max_depth': [None, 3, ...], 'max_features': [None, 'sqrt', ...], ...}"
,n_iter,60
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [ ]:
print("Mejores parámetros:", random_tree.best_params_)

best_model_tree = random_tree.best_estimator_

pred_tree = best_model_tree.predict(x_test_scaled)

accuracy_tree = best_model_tree.score(x_test_scaled, y_test)
print(f"La precisión del modelo es: {accuracy_tree:.2f}")

Mejores parámetros: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 19, 'max_features': 'sqrt', 'min_samples_leaf': 9, 'min_samples_split': 27}
La precisión del modelo es: 0.64
